# CYK Parser Evaluation on Labeled Sample

This notebook evaluates the CYK parser on the `utterances_sample_labeled.csv` dataset and analyzes failure cases grouped by section label.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import nltk
from nltk import bigrams, trigrams

from src import HearingLoader, HearingTagger
from src.grammar.Parser import Parser
from src.dataclasses.Hearing import Hearing
from src.dataclasses.OralContribution import OralContribution
from src.speakers.Speaker import Speaker
from src.speakers.enums.SpeakerPositionEnum import SpeakerPositionEnum
from datetime import datetime

## Load Labeled Sample Data

In [2]:
# Load the labeled sample CSV
df = pd.read_csv('utterances_sample_labeled.csv')

print(f"Total utterances: {len(df)}")
print(f"Unique hearings: {df['hid'].nunique()}")
print(f"Unique bills: {df['bid'].nunique()}")
print(f"\nSection labels:")
print(df['section'].value_counts())

Total utterances: 2619
Unique hearings: 94
Unique bills: 100

Section labels:
section
PUBLIC_COMMENTS          766
LEGISLATOR_DISCUSSION    739
VOTE                     270
EXPERT_TESTIMONY         241
CLOSING_REMARKS          186
PRESENTATION             129
OTHER_HEARING             81
OTHER_PROCEDURAL          80
INTRO                     79
OTHER_NONPROCEDURAL       48
Name: count, dtype: int64


## Reconstruct Hearings from CSV

In [3]:
# Group by hearing (hid + bid)
grouped = df.groupby(['hid', 'bid'])

hearings = []

for (hid, bid), group in grouped:
    # Get hearing-level metadata from first row
    first_row = group.iloc[0]
    state = first_row['state']
    
    # Create speakers dictionary from unique pids in the group
    speakers = {}
    for pid in group['pid'].unique():
        pid_rows = group[group['pid'] == pid]
        speaker_position_str = pid_rows.iloc[0]['speaker.position']
        
        # Convert speaker position string to enum
        try:
            speaker_position = SpeakerPositionEnum[speaker_position_str]
        except (KeyError, AttributeError):
            speaker_position = None
        
        # Find first and last uid for this speaker
        speaker_uids = pid_rows['uid'].values
        first_uid = int(speaker_uids.min())
        last_uid = int(speaker_uids.max())
        
        speakers[pid] = Speaker(
            pid=pid,
            first_name=None,
            last_name=None,
            speaker_position=speaker_position,
            can_file_motions=None,
            is_presenter=None,
            first_mention_uid=None,
            first_uid=first_uid,
            last_uid=last_uid
        )
    
    # Create utterances
    utterances = []
    for _, row in group.iterrows():
        utt = OralContribution(
            uid=int(row['uid']),
            pid=int(row['pid']),
            text=str(row['text'])
        )
        # Store the section label as an attribute for later reference
        utt.section = row['section']
        utterances.append(utt)
    
    # Create hearing with minimal required fields
    # We don't have all metadata from CSV, so use defaults
    hearing = Hearing(
        state=state,
        bid=bid,
        hid=int(hid),
        cid=0,  # Not in CSV
        cname="Unknown",  # Not in CSV
        hearing_date=datetime(2015, 1, 1),  # Not in CSV
        speakers=speakers,
        utterances=utterances
    )
    hearings.append(hearing)

print(f"\nReconstructed {len(hearings)} hearings from CSV")


Reconstructed 100 hearings from CSV


## Initialize Parser and Tag Hearings

In [4]:
tagger = HearingTagger()
parser = Parser()

print("Tagging hearings...")
tagged_hearings = []
tagging_errors = []

for hearing in hearings:
    try:
        # Preserve original section labels before tagging
        original_sections = [getattr(utt, 'section', None) for utt in hearing.utterances]
        
        tagged = tagger(hearing)
        if tagged:
            # Restore section labels to tagged utterances
            for i, utt in enumerate(tagged.utterances):
                if i < len(original_sections):
                    utt.section = original_sections[i]
            tagged_hearings.append(tagged)
    except Exception as e:
        tagging_errors.append((hearing.hid, hearing.bid, str(e)))

print(f"Successfully tagged: {len(tagged_hearings)}/{len(hearings)}")
print(f"Tagging errors: {len(tagging_errors)}")

if tagging_errors:
    print("\nTagging errors:")
    for hid, bid, error in tagging_errors[:5]:
        print(f"  - Hearing {hid}, Bill {bid}: {error[:100]}")

Tagging hearings...
Successfully tagged: 62/100
Tagging errors: 38

Tagging errors:
  - Hearing 51835, Bill CA_201720180AB37: min() iterable argument is empty
  - Hearing 52102, Bill CA_201720180AB665: min() iterable argument is empty
  - Hearing 52381, Bill CA_201720180AB76: min() iterable argument is empty
  - Hearing 52464, Bill CA_201720180AB321: min() iterable argument is empty
  - Hearing 52465, Bill CA_201720180AB389: min() iterable argument is empty


## Evaluate Parser on All Hearings

In [5]:
print("Parsing all hearings...")
print("=" * 80)

parse_results = []
successful_parses = []
failed_parses = []

for i, hearing in enumerate(tagged_hearings):
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(tagged_hearings)} hearings")
    
    try:
        parse_trees = list(parser.get_all_parses_as_nltk_trees(hearing, max_parses=1) or [])
        parse_success = len(parse_trees) > 0
        
        parse_results.append(int(parse_success))
        
        if parse_success:
            successful_parses.append((hearing.hid, hearing.bid, hearing))
        else:
            failed_parses.append((hearing.hid, hearing.bid, hearing))
    except Exception as e:
        parse_results.append(0)
        failed_parses.append((hearing.hid, hearing.bid, hearing))

print(f"\nParsing complete!")
print(f"Successfully processed: {len(parse_results)}/{len(tagged_hearings)} hearings")

Parsing all hearings...
Processed 10/62 hearings
Processed 20/62 hearings
Processed 30/62 hearings
Processed 40/62 hearings
Processed 50/62 hearings
Processed 60/62 hearings

Parsing complete!
Successfully processed: 62/62 hearings


## Parser Success Rate Statistics

In [6]:
total_hearings = len(parse_results)
successful_count = np.sum(parse_results)
failed_count = total_hearings - successful_count
success_rate = np.mean(parse_results) * 100 if parse_results else 0

print("=" * 80)
print("CYK Parser Statistics for Labeled Sample")
print("=" * 80)
print(f"\nTotal hearings evaluated: {total_hearings}")
print(f"Successful parses: {successful_count}")
print(f"Failed parses: {failed_count}")
print(f"\nSuccess rate: {success_rate:.2f}%")
print("\n" + "=" * 80)

CYK Parser Statistics for Labeled Sample

Total hearings evaluated: 62
Successful parses: 48
Failed parses: 14

Success rate: 77.42%



## Analyze Failed Parses by Section Label

In [7]:
print("Analyzing failed parses by section label...")
print("=" * 80)

# Group failed hearing tokens by section
section_tokens = defaultdict(list)
section_hearings = defaultdict(list)

for hid, bid, hearing in failed_parses:
    tokens = parser.tokenizer.tokenize_utterances(hearing)
    
    # Get token sequence with section labels
    # tokens is List[(SpeakerPositionEnum, utterance_index)]
    for speaker_position, utt_idx in tokens:
        token_name = speaker_position.name if speaker_position else 'UNKNOWN'
        utterance = hearing.utterances[utt_idx]
        section_label = getattr(utterance, 'section', None)
        if section_label is None or section_label == '' or pd.isna(section_label):
            section_label = 'UNKNOWN'
        
        # Store token by section
        section_tokens[section_label].append(token_name)
        
    # Store hearing info
    for utt in hearing.utterances:
        section_label = getattr(utt, 'section', None)
        if section_label is None or section_label == '' or pd.isna(section_label):
            section_label = 'UNKNOWN'
        section_hearings[section_label].append((hid, bid, hearing))

# Display section statistics
print(f"\nTotal sections in failed parses: {len(section_tokens)}")
print("\nToken counts by section:")
for section in sorted(section_tokens.keys()):
    token_count = len(section_tokens[section])
    hearing_count = len(set((h, b) for h, b, _ in section_hearings[section]))
    section_str = str(section) if section else 'UNKNOWN'
    print(f"  {section_str:30s}: {token_count:4d} tokens across {hearing_count} hearings")

Analyzing failed parses by section label...

Total sections in failed parses: 10

Token counts by section:
  CLOSING_REMARKS               :   33 tokens across 13 hearings
  EXPERT_TESTIMONY              :   37 tokens across 10 hearings
  INTRO                         :   11 tokens across 10 hearings
  LEGISLATOR_DISCUSSION         :  185 tokens across 14 hearings
  OTHER_HEARING                 :   13 tokens across 4 hearings
  OTHER_NONPROCEDURAL           :   10 tokens across 6 hearings
  OTHER_PROCEDURAL              :   26 tokens across 9 hearings
  PRESENTATION                  :   27 tokens across 13 hearings
  PUBLIC_COMMENTS               :  168 tokens across 14 hearings
  VOTE                          :   62 tokens across 13 hearings


## N-gram Analysis by Section Label

Analyze bigrams and trigrams for each section label in failed parses.

In [8]:
# Analyze bigrams and trigrams for each section
section_bigrams = {}
section_trigrams = {}

for section, tokens in section_tokens.items():
    if len(tokens) >= 2:
        bigram_list = list(bigrams(tokens))
        section_bigrams[section] = Counter(bigram_list)
    
    if len(tokens) >= 3:
        trigram_list = list(trigrams(tokens))
        section_trigrams[section] = Counter(trigram_list)

print("N-gram analysis complete!")
print(f"Sections with bigrams: {len(section_bigrams)}")
print(f"Sections with trigrams: {len(section_trigrams)}")

N-gram analysis complete!
Sections with bigrams: 10
Sections with trigrams: 10


## Summary Report

In [9]:
print("=" * 80)
print("SUMMARY REPORT")
print("=" * 80)

print(f"\nDataset: utterances_sample_labeled.csv")
print(f"Total hearings evaluated: {len(tagged_hearings)}")
print(f"Successful parses: {successful_count} ({success_rate:.2f}%)")
print(f"Failed parses: {failed_count} ({100-success_rate:.2f}%)")

print(f"\nFailed Parse Analysis:")
print(f"  - Unique sections in failures: {len(section_tokens)}")
print(f"  - Total tokens in failures: {sum(len(tokens) for tokens in section_tokens.values())}")

print("\n" + "=" * 80)

SUMMARY REPORT

Dataset: utterances_sample_labeled.csv
Total hearings evaluated: 62
Successful parses: 48 (77.42%)
Failed parses: 14 (22.58%)

Failed Parse Analysis:
  - Unique sections in failures: 10
  - Total tokens in failures: 572



## Comparison: Successful vs Failed Parse N-gram Frequencies

Compare bigram and trigram occurrence frequencies between successful and failed parses to identify patterns that distinguish them.

In [10]:
# Extract tokens from successful parses
print("Analyzing successful parses...")
successful_tokens = []

for hid, bid, hearing in successful_parses:
    tokens = parser.tokenizer.tokenize_utterances(hearing)
    token_sequence = [speaker_position.name if speaker_position else 'UNKNOWN' 
                     for speaker_position, utt_idx in tokens]
    successful_tokens.extend(token_sequence)

# Extract tokens from failed parses (already have this from earlier analysis)
failed_tokens = []
for section, tokens in section_tokens.items():
    failed_tokens.extend(tokens)

print(f"Total tokens in successful parses: {len(successful_tokens)}")
print(f"Total tokens in failed parses: {len(failed_tokens)}")

# Compute bigrams and trigrams for successful parses
successful_bigrams = Counter(list(bigrams(successful_tokens)))
successful_trigrams = Counter(list(trigrams(successful_tokens)))

# Compute bigrams and trigrams for failed parses
failed_bigrams = Counter(list(bigrams(failed_tokens)))
failed_trigrams = Counter(list(trigrams(failed_tokens)))

print(f"\nSuccessful parses:")
print(f"  - Unique bigrams: {len(successful_bigrams)}")
print(f"  - Unique trigrams: {len(successful_trigrams)}")

print(f"\nFailed parses:")
print(f"  - Unique bigrams: {len(failed_bigrams)}")
print(f"  - Unique trigrams: {len(failed_trigrams)}")

# Compute total counts for percentage calculations
total_successful_bigrams = sum(successful_bigrams.values())
total_successful_trigrams = sum(successful_trigrams.values())
total_failed_bigrams = sum(failed_bigrams.values())
total_failed_trigrams = sum(failed_trigrams.values())

Analyzing successful parses...
Total tokens in successful parses: 1553
Total tokens in failed parses: 572

Successful parses:
  - Unique bigrams: 28
  - Unique trigrams: 99

Failed parses:
  - Unique bigrams: 30
  - Unique trigrams: 86


### Statistical Analysis: Unique Patterns

Identify bigrams and trigrams that appear exclusively or predominantly in one group.

In [11]:
# Find bigrams and trigrams that appear only in failed parses, grouped by section
failed_only_bigrams_by_section = {}
failed_only_trigrams_by_section = {}

for section in section_tokens.keys():
    # Get bigrams and trigrams for this section
    section_bg = section_bigrams.get(section, Counter())
    section_tg = section_trigrams.get(section, Counter())
    
    # Find exclusive patterns for this section
    failed_only_bigrams_by_section[section] = {
        bg: count for bg, count in section_bg.items() 
        if bg not in successful_bigrams
    }
    
    failed_only_trigrams_by_section[section] = {
        tg: count for tg, count in section_tg.items() 
        if tg not in successful_trigrams
    }

# Overall exclusive patterns
all_failed_bigrams = Counter()
all_failed_trigrams = Counter()
for section_bg in section_bigrams.values():
    all_failed_bigrams.update(section_bg)
for section_tg in section_trigrams.values():
    all_failed_trigrams.update(section_tg)

successful_only_bigrams = {bg: count for bg, count in successful_bigrams.items() 
                           if bg not in all_failed_bigrams}
failed_only_bigrams = {bg: count for bg, count in all_failed_bigrams.items() 
                       if bg not in successful_bigrams}

successful_only_trigrams = {tg: count for tg, count in successful_trigrams.items() 
                            if tg not in all_failed_trigrams}
failed_only_trigrams = {tg: count for tg, count in all_failed_trigrams.items() 
                        if tg not in successful_trigrams}

print("=" * 80)
print("EXCLUSIVE PATTERNS ANALYSIS")
print("=" * 80)

print(f"\nBigrams appearing ONLY in successful parses: {len(successful_only_bigrams)}")
print(f"Bigrams appearing ONLY in failed parses: {len(failed_only_bigrams)}")
print(f"Trigrams appearing ONLY in successful parses: {len(successful_only_trigrams)}")
print(f"Trigrams appearing ONLY in failed parses: {len(failed_only_trigrams)}")

# Display exclusive patterns grouped by section
print("\n" + "=" * 80)
print("PATTERNS EXCLUSIVE TO FAILED PARSES BY SECTION LABEL")
print("=" * 80)

for section in sorted(section_tokens.keys()):
    section_only_bg = failed_only_bigrams_by_section[section]
    section_only_tg = failed_only_trigrams_by_section[section]
    
    if not section_only_bg and not section_only_tg:
        continue
    
    print(f"\n{section}")
    print("-" * 80)
    
    if section_only_bg:
        print(f"\n  Exclusive Bigrams ({len(section_only_bg)}):")
        for bg, count in sorted(section_only_bg.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"    {bg[0]:20s} -> {bg[1]:20s}  (count: {count})")
    
    if section_only_tg:
        print(f"\n  Exclusive Trigrams ({len(section_only_tg)}):")
        for tg, count in sorted(section_only_tg.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"    {tg[0]:20s} -> {tg[1]:20s} -> {tg[2]:20s}  (count: {count})")

EXCLUSIVE PATTERNS ANALYSIS

Bigrams appearing ONLY in successful parses: 2
Bigrams appearing ONLY in failed parses: 4
Trigrams appearing ONLY in successful parses: 40
Trigrams appearing ONLY in failed parses: 24

PATTERNS EXCLUSIVE TO FAILED PARSES BY SECTION LABEL

EXPERT_TESTIMONY
--------------------------------------------------------------------------------

  Exclusive Bigrams (1):
    EXPERT               -> SECRETARY             (count: 1)

  Exclusive Trigrams (4):
    EXPERT               -> EXPERT               -> EXPERT                (count: 2)
    EXPERT               -> EXPERT               -> SECRETARY             (count: 1)
    EXPERT               -> SECRETARY            -> PRESIDING_CHAIR       (count: 1)
    SECRETARY            -> PRESIDING_CHAIR      -> EXPERT                (count: 1)

LEGISLATOR_DISCUSSION
--------------------------------------------------------------------------------

  Exclusive Trigrams (4):
    COMMITTEE_MEMBER     -> EXPERT               